# Clase 141 — Encoder-Decoder para traducción

La arquitectura **seq2seq** (Sutskever et al. 2014): un **encoder** comprime la
oración fuente en un estado de contexto y un **decoder** genera la traducción token
a token. Vemos **teacher forcing**, tokens `[start]`/`[end]`, y **BLEU**.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`, `nltk` (para BLEU).

## 1. Pares fuente-destino y tokens especiales

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
pares = [
    ("hello", "hola"),
    ("good morning", "buenos dias"),
    ("thank you", "gracias"),
    ("how are you", "como estas"),
    ("see you later", "hasta luego"),
]
fuente = [s for s, _ in pares]
destino = ["[start] " + t + " [end]" for _, t in pares]   # marcadores de inicio/fin
print(destino[0])

## 2. Vectorizadores independientes para fuente y destino

In [ ]:
VOCAB, LEN = 1000, 8
vec_src = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=LEN)
vec_tgt = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=LEN + 1)
vec_src.adapt(fuente)
vec_tgt.adapt(destino)
print("vocab destino:", len(vec_tgt.get_vocabulary()))

## 3. Encoder: `LSTM(return_state=True)` → estado de contexto

In [ ]:
DIM = 256
enc_in = keras.Input(shape=(1,), dtype="string")
ex = vec_src(enc_in)
ex = layers.Embedding(VOCAB, DIM, mask_zero=True)(ex)
_, state_h, state_c = layers.LSTM(DIM, return_state=True)(ex)   # (h_T, c_T) = contexto
encoder = keras.Model(enc_in, [state_h, state_c], name="encoder")
encoder.summary()

## 4. Decoder con teacher forcing (`initial_state` = estado del encoder)

In [ ]:
dec_in = keras.Input(shape=(1,), dtype="string")
dx = vec_tgt(dec_in)
dx = layers.Embedding(VOCAB, DIM, mask_zero=True)(dx)
dx = layers.LSTM(DIM, return_sequences=True)(dx, initial_state=[state_h, state_c])
salida = layers.Dense(VOCAB, activation="softmax")(dx)
seq2seq = keras.Model([enc_in, dec_in], salida, name="seq2seq")
seq2seq.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
# Teacher forcing: en training el decoder ve el target real (shifted), no su predicción.
seq2seq.summary()

## 5. Inference autoregresiva (bucle `[start]` → … → `[end]`)

In [ ]:
vocab_tgt = vec_tgt.get_vocabulary()
def traducir(texto, max_len=LEN):
    h, c = encoder.predict(np.array([texto]), verbose=0)   # contexto de la fuente
    generado = "[start]"
    gen = np.random.default_rng(0)
    for _ in range(max_len):
        # Ilustrativo: en un decoder real se reusa el estado y se realimenta la predicción.
        siguiente = vocab_tgt[gen.integers(2, len(vocab_tgt))]
        if siguiente == "[end]":
            break
        generado += " " + siguiente
    return generado.replace("[start]", "").strip()

print("traducción (modelo sin entrenar, ilustrativo):", traducir("hello"))

## 6. Evaluar con BLEU

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
suave = SmoothingFunction().method1
ref = [["hola", "buenos", "dias"]]
print("BLEU perfecto:", round(sentence_bleu(ref, ["hola", "buenos", "dias"], smoothing_function=suave), 3))
print("BLEU parcial :", round(sentence_bleu(ref, ["hola", "tardes"], smoothing_function=suave), 3))

## Ejercicios

1. **Preparar datos**: tokenizá fuente y destino, agregá `[start]`/`[end]` y padding.
2. **Encoder**: `Embedding → LSTM(256, return_state=True)`, conservá `state_h`, `state_c`.
3. **Decoder en training**: teacher forcing con `initial_state = encoder_state`.
4. **BLEU**: calculá BLEU sobre un test set con `nltk` y `SmoothingFunction`.

## Conclusiones

- **seq2seq**: encoder resume la fuente en `(h_T, c_T)`; decoder genera desde ese estado.
- **Teacher forcing** acelera el training (target real como input) pero crea mismatch con inference.
- Los tokens `[start]`/`[end]` marcan el inicio y el fin de la generación.
- El **cuello de botella** (todo el significado en un vector fijo) motivó la **atención** (clase 142).
- **BLEU** mide solapamiento de n-gramas (0-100); es aceptable pero falla con paráfrasis.